# Skin Cancer Classification with PyTorch
This notebook builds a PyTorch training pipeline using the HAM10000 dataset. It reuses the same metadata and image loading logic as the existing TensorFlow notebook, but trains a ResNet-50 model in PyTorch.

In [ ]:
import os
import glob
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

import torch
from torch import nn, optim
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from torchvision.utils import make_grid

import kagglehub

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
path = kagglehub.dataset_download('kmader/skin-cancer-mnist-ham10000')
print('Dataset path:', path)

META_PATH = os.path.join(path, 'HAM10000_metadata.csv')
IMG_DIRS = [
    os.path.join(path, 'HAM10000_images_part_1'),
    os.path.join(path, 'HAM10000_images_part_2'),
]

df = pd.read_csv(META_PATH)
print('Rows loaded:', len(df))
df.head()

In [ ]:
LABEL_MAP = {
    'akiec': 'Actinic Keratoses / Intraepithelial Carcinoma',
    'bcc': 'Basal Cell Carcinoma',
    'bkl': 'Benign Keratosis',
    'df': 'Dermatofibroma',
    'mel': 'Melanoma',
    'nv': 'Melanocytic Nevi',
    'vasc': 'Vascular Lesions',
}

MALIGNANT = {'mel', 'bcc', 'akiec'}

image_paths = {}
for d in IMG_DIRS:
    for fp in glob.glob(os.path.join(d, '*.jpg')):
        image_id = os.path.splitext(os.path.basename(fp))[0]
        image_paths[image_id] = fp

df['image_path'] = df['image_id'].map(image_paths)
df['dx_full'] = df['dx'].map(LABEL_MAP)
df['malignant'] = df['dx'].isin(MALIGNANT)

print('Missing image paths:', df['image_path'].isna().sum())
df = df.dropna(subset=['image_path']).reset_index(drop=True)
print('Final rows:', len(df))
print(df['dx'].value_counts())

In [ ]:
CLASS_NAMES = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
class_to_index = {cls: i for i, cls in enumerate(CLASS_NAMES)}
index_to_class = {i: cls for cls, i in class_to_index.items()}

df = df[df['dx'].isin(CLASS_NAMES)].copy()
df['label'] = df['dx'].map(class_to_index)

print('Class mapping:', class_to_index)
df[['dx', 'label']].head()

In [ ]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df['label'],
    random_state=SEED,
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df['label'],
    random_state=SEED,
)

print('Train size:', len(train_df))
print('Val size:', len(val_df))
print('Test size:', len(test_df))

print('Train distribution:')
print(train_df['dx'].value_counts())

In [ ]:
IMG_HEIGHT = 224
IMG_WIDTH = 224
BATCH_SIZE = 192
NUM_CLASSES = len(CLASS_NAMES)

train_transform = transforms.Compose([
    transforms.Resize((IMG_HEIGHT, IMG_WIDTH)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_HEIGHT, IMG_WIDTH)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

In [ ]:
class SkinCancerDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        image = Image.open(row['image_path']).convert('RGB')
        if self.transform is not None:
            image = self.transform(image)
        label = torch.tensor(row['label'], dtype=torch.long)
        return image, label

train_dataset = SkinCancerDataset(train_df, transform=train_transform)
val_dataset = SkinCancerDataset(val_df, transform=eval_transform)
test_dataset = SkinCancerDataset(test_df, transform=eval_transform)

# In Windows notebooks, worker subprocesses can hang; keep workers at 0 there.
NUM_WORKERS = 0 if os.name == 'nt' else 2
PIN_MEMORY = torch.cuda.is_available()
NON_BLOCKING = device.type == 'cuda'

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=NUM_WORKERS > 0,
    )
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=NUM_WORKERS > 0,
    )
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=NUM_WORKERS > 0,
    )

In [ ]:
def show_batch(loader, class_names):
    images, labels = next(iter(loader))
    grid = make_grid(images[:8], nrow=4, normalize=True)
    plt.figure(figsize=(10, 6))
    plt.imshow(grid.permute(1, 2, 0).detach().cpu())
    plt.axis('off')
    plt.title('Sample Augmented Images')
    plt.show()

show_batch(train_loader, CLASS_NAMES)

In [ ]:
torch.set_float32_matmul_precision('high')

model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
for param in model.parameters():
    param.requires_grad = False

num_ftrs = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(0.4),
    nn.Linear(num_ftrs, 256),
    nn.ReLU(inplace=True),
    nn.Dropout(0.3),
    nn.Linear(256, NUM_CLASSES)
)
model = model.to(device, memory_format=torch.channels_last)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.3, patience=2, min_lr=1e-6)

In [ ]:
torch.backends.cudnn.benchmark = True
USE_AMP = device.type == 'cuda'
scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)

def run_epoch(loader, model, criterion, optimizer=None):
    running_loss = 0.0
    running_corrects = 0
    total = 0
    training = optimizer is not None

    if training:
        model.train()
    else:
        model.eval()

    for images, labels in loader:
        images = images.to(device, non_blocking=NON_BLOCKING, memory_format=torch.channels_last)
        labels = labels.to(device, non_blocking=NON_BLOCKING)

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_AMP):
            outputs = model(images)
            loss = criterion(outputs, labels)

        if training:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        _, preds = torch.max(outputs, 1)
        running_loss += loss.item() * images.size(0)
        running_corrects += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, running_corrects / total

EPOCHS = 20
best_val_loss = float('inf')
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_epoch(train_loader, model, criterion, optimizer)
    val_loss, val_acc = run_epoch(val_loader, model, criterion)

    scheduler.step(val_loss)

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    print(f'Epoch {epoch}/{EPOCHS} - train_loss: {train_loss:.4f}, train_acc: {train_acc:.4f}, val_loss: {val_loss:.4f}, val_acc: {val_acc:.4f}')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'pytorch_skin_cancer_resnet50.pth')

print('Training complete. Best val loss:', best_val_loss)

In [ ]:
epochs_range = range(1, len(history['train_acc']) + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs_range, history['train_acc'], label='Training Accuracy')
plt.plot(epochs_range, history['val_acc'], label='Validation Accuracy')
plt.title('Training vs Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(epochs_range, history['train_loss'], label='Training Loss')
plt.plot(epochs_range, history['val_loss'], label='Validation Loss')
plt.title('Training vs Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
model.load_state_dict(torch.load('pytorch_skin_cancer_resnet50.pth', map_location=device))
model.eval()

y_true = []
y_pred = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device, non_blocking=NON_BLOCKING)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print(classification_report(y_true, y_pred, target_names=[LABEL_MAP[c] for c in CLASS_NAMES], digits=4))

In [ ]:
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(10, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[LABEL_MAP[c] for c in CLASS_NAMES])
disp.plot(ax=ax, xticks_rotation=45, colorbar=False)
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()